In [2]:
# Amanda Rodgers
# Week 5 Homework
# 09/2/26

In [3]:
# IMPORT LIBRARIES
import requests
import pandas as pd
import numpy as np

In [4]:
# Download ETH data from Coingecko

# This is the URL for CoinGecko's Ethereum market data API.
url = "https://api.coingecko.com/api/v3/coins/ethereum/market_chart"


# These are the settings we send with our API request.
params = {
    "vs_currency": "usd",  # Get Bitcoin prices in US Dollars
    "days": "180",         # Get the last 180 days of data
    "interval": "daily"    # Get one data point per day
}


# requests.get() sends a request to the API.
# The API then sends data back to Python.
response = requests.get(url, params=params)


# raise_for_status() checks if the API request was successful.
# If there is an error, Python will display the error instead
# of continuing with incorrect or missing data.
response.raise_for_status()


# .json() converts the API response into Python data.
# The result is usually a dictionary containing lists.
data = response.json()

In [5]:
# Create Pandas dataframe

# The API gives us price data in this format:
#
# [timestamp, price]
#
# Example:
# [1725148800000, 59000]
#
# pd.DataFrame() converts this data into a table.
prices = pd.DataFrame(
    data["prices"],
    columns=["timestamp", "Close"]
)

# "timestamp" = the date and time in milliseconds
# "Close" = the Bitcoin price


# The API gives us volume data in this format:
#
# [timestamp, volume]
#
# We also convert this into a DataFrame.
volumes = pd.DataFrame(
    data["total_volumes"],
    columns=["timestamp", "Volume"]
)

# "Volume" represents the amount of Ethereum trading activity.

In [6]:
# Convert timestamp into real date

# pd.to_datetime() converts timestamps into readable dates.
#
# unit="ms" means the timestamp is measured in milliseconds.
prices["Date"] = pd.to_datetime(
    prices["timestamp"],
    unit="ms"
)


# We do the same thing for the volume DataFrame.
volumes["Date"] = pd.to_datetime(
    volumes["timestamp"],
    unit="ms"
)

In [7]:
# Combine price and volume dataframes

# merge() combines two DataFrames together.
#
# We combine them using the "Date" column.
#
# on="Date" tells pandas which column to use when matching data.
df = prices.merge(
    volumes,
    on="Date"
)


# Select only the columns we want to keep.
#
# We do not need the original timestamp columns anymore.
df = df[["Date", "Close", "Volume"]]


# sort_values() puts the data in chronological order.
#
# ascending=True means oldest dates come first.
df = df.sort_values(
    "Date",
    ascending=True
)


# set_index() makes the Date column the index of the DataFrame.
#
# An index helps us work with time-series data.
df = df.set_index("Date")


# Display the first 5 rows of the original data.
#
# head() shows the first 5 rows by default.
print("RAW ETHEREUM DATA")
print(df.head())

RAW ETHEREUM DATA
                  Close        Volume
Date                                 
2026-03-07  1979.078133  1.788093e+10
2026-03-08  1984.447969  1.310771e+09
2026-03-09  1939.391759  1.514419e+10
2026-03-10  1992.982027  2.150956e+10
2026-03-11  2035.174154  2.043350e+10


In [8]:
# Calculate daily return
df["daily_return"] = df["Close"].pct_change()

# View df
print(df.head())

                  Close        Volume  daily_return
Date                                               
2026-03-07  1979.078133  1.788093e+10           NaN
2026-03-08  1984.447969  1.310771e+09      0.002713
2026-03-09  1939.391759  1.514419e+10     -0.022705
2026-03-10  1992.982027  2.150956e+10      0.027633
2026-03-11  2035.174154  2.043350e+10      0.021170


In [9]:
# Move data down a row to compare today's price with yesterday's price
# Calculate log return

df["log_return"] = np.log(
    df["Close"] / df["Close"].shift(1)
)

print(df.head())

                  Close        Volume  daily_return  log_return
Date                                                           
2026-03-07  1979.078133  1.788093e+10           NaN         NaN
2026-03-08  1984.447969  1.310771e+09      0.002713    0.002710
2026-03-09  1939.391759  1.514419e+10     -0.022705   -0.022966
2026-03-10  1992.982027  2.150956e+10      0.027633    0.027258
2026-03-11  2035.174154  2.043350e+10      0.021170    0.020949


In [10]:
# Create moving averages
# rolling(7) creates a moving window of 7 days.
#
# mean() calculates the average.
#
# This gives us the average Bitcoin price over the last 7 days.
df["ma_7"] = df["Close"].rolling(7).mean()


# Average Bitcoin price over the last 14 days.
df["ma_14"] = df["Close"].rolling(14).mean()


# Average Bitcoin price over the last 30 days.
df["ma_30"] = df["Close"].rolling(30).mean()

In [11]:
# Calcualte volatility with STD

# rolling(7).std() calculates the standard deviation
# of the last 7 prices.
#
# Standard deviation measures how much the prices are changing.
#
# A higher value generally means the market is more volatile.
df["volatility_7"] = df["Close"].rolling(7).std()


# Calculate volatility using the previous 14 days.
df["volatility_14"] = df["Close"].rolling(14).std()

In [12]:
# Calculate percent change in trading volume

# Calculate the percentage change in trading volume.
#
# pct_change() compares today's volume to yesterday's volume.
df["volume_change"] = df["Volume"].pct_change()


# Calculate the average trading volume over the last 7 days.
df["avg_volume_7"] = df["Volume"].rolling(7).mean()

In [13]:
# Calculate volume spike
# np.where(condition, value_if_true, value_if_false)
df["volume_spike"] = np.where(
    df["Volume"] > 1.5 * df["avg_volume_7"],
    1,
    0
)

In [14]:
# Calculate RSI, if price change negative print 0, if positive print 1

# diff() calculates the difference between today's closing
# price and yesterday's closing price.
#
# Positive number = price increased
# Negative number = price decreased
df["price_change"] = df["Close"].diff()


# Create a column containing only positive price changes.
#
# where() keeps the value when the condition is True.
#
# If the price change is not positive, we replace it with 0.
df["gain"] = df["price_change"].where(
    df["price_change"] > 0,
    0
)


# Create a column containing only price losses.
#
# First, we find negative price changes.
# Then we multiply by -1 to make the loss positive.
df["loss"] = -df["price_change"].where(
    df["price_change"] < 0,
    0
)


# Calculate the average gain over the last 14 days.
df["avg_gain"] = df["gain"].rolling(14).mean()


# Calculate the average loss over the last 14 days.
df["avg_loss"] = df["loss"].rolling(14).mean()


# Calculate Relative Strength (RS).
#
# RS = Average Gain / Average Loss
rs = df["avg_gain"] / df["avg_loss"]


# Calculate the 14-day RSI.
#
# RSI formula:
#
# RSI = 100 - (100 / (1 + RS))
#
# RSI generally ranges from 0 to 100.
#
# Higher RSI can suggest strong upward momentum.
# Lower RSI can suggest strong downward momentum.
df["rsi_14"] = 100 - (
    100 / (1 + rs)
)


In [15]:
# Add lag features

# A lag feature gives us information from previous days.
#
# Lag features are useful for machine learning because
# today's market behavior may be influenced by previous data.


# Bitcoin closing price from 1 day ago.
df["close_lag_1"] = df["Close"].shift(1)


# Bitcoin closing price from 3 days ago.
df["close_lag_3"] = df["Close"].shift(3)


# Bitcoin closing price from 7 days ago.
df["close_lag_7"] = df["Close"].shift(7)


# Daily return from 1 day ago.
df["return_lag_1"] = df["daily_return"].shift(1)


# Daily return from 3 days ago.
df["return_lag_3"] = df["daily_return"].shift(3)

In [16]:
# CREATE TARGET VARIABLE

# Our machine learning model will try to predict:
#
# Will Bitcoin's price go UP tomorrow?
#
# 1 = Yes, the price goes up
# 0 = No, the price does not go up


# shift(-1) moves the Close price UP one row.
#
# This allows us to compare today's price with tomorrow's price.
df["price_up_tomorrow"] = np.where(
    df["Close"].shift(-1) > df["Close"],
    1,
    0
)

In [17]:
# Remove rows with NaN ( missing values )

# dropna() removes rows containing missing values.
df = df.dropna()

In [18]:
# Remove unneeded columns

# The following columns were only needed to calculate RSI.
#
# We remove them to keep our final machine learning dataset
# clean and simple.
df = df.drop(
    columns=[
        "price_change",
        "gain",
        "loss",
        "avg_gain",
        "avg_loss"
    ]
)

In [19]:
########################################################
# ETHEREUM DATA: FEATURE ENGINEERING AND DATA VISUALIZATION
###########################################################

In [20]:
# Import Libraries
import requests
# requests allows Python to connect to APIs and download data.

import pandas as pd
# pandas allows us to organize and analyze data using DataFrames.

import numpy as np
# numpy is used for numerical calculations.

import matplotlib.pyplot as plt
# matplotlib.pyplot is used to create charts and visualizations.

In [21]:
# DOWNLOAD ETH DATA FROM COINGECKO
# We are specifically requesting data for Bitcoin.
url = "https://api.coingecko.com/api/v3/coins/ethereum/market_chart"


# These parameters tell the API what data we want.
params = {

    # Get Bitcoin prices in US Dollars.
    "vs_currency": "usd",

    # Get the last 180 days of data.
    "days": "180",

    # Return the data using daily intervals.
    "interval": "daily"
}


# requests.get() sends a request to the CoinGecko API.
#
# The API will send the Bitcoin data back to Python.
response = requests.get(url, params=params)


# raise_for_status() checks whether the API request worked.
#
# If the API returns an error, Python will stop and display
# information about the problem.
response.raise_for_status()


# .json() converts the API response into Python data.
#
# The result will contain dictionaries and lists that we can
# work with in Python.
data = response.json()


In [22]:
# Create dataframes

# The price data from CoinGecko looks similar to this:
#
# [timestamp, price]
#
# We convert the price data into a pandas DataFrame.
prices = pd.DataFrame(
    data["prices"],
    columns=["timestamp", "Close"]
)


# The volume data looks similar to this:
#
# [timestamp, volume]
#
# We convert the volume data into another DataFrame.
volumes = pd.DataFrame(
    data["total_volumes"],
    columns=["timestamp", "Volume"]
)

In [23]:
# CONVERT TIMESTAMPS TO DATES

# The API provides timestamps in milliseconds.
#
# pd.to_datetime() converts those timestamps into readable dates.
#
# unit="ms" tells pandas that the timestamps are in milliseconds.
prices["Date"] = pd.to_datetime(
    prices["timestamp"],
    unit="ms"
)


# Convert the volume timestamps into dates as well.
volumes["Date"] = pd.to_datetime(
    volumes["timestamp"],
    unit="ms"
)

In [24]:
# COMBINE PRICE AND VOLUME DATAFRAMES

# merge() combines two DataFrames.
#
# We combine the price and volume DataFrames using the Date column.
df = prices.merge(
    volumes,
    on="Date"
)


# Keep only the columns we need.
#
# The timestamp columns are no longer needed.
df = df[["Date", "Close", "Volume"]]


# sort_values() organizes the data by date.
#
# Oldest dates will appear first.
df = df.sort_values("Date")


# set_index() makes the Date column the index.
#
# This is useful when working with time-series data
# and when creating charts.
df = df.set_index("Date")


# Display the first five rows of the raw data.
#
# head() shows the first five rows by default.
print("RAW ETHEREUM DATA")
print(df.head())

RAW ETHEREUM DATA
                  Close        Volume
Date                                 
2026-03-07  1979.078133  1.788093e+10
2026-03-08  1984.447969  1.310771e+09
2026-03-09  1939.391759  1.514419e+10
2026-03-10  1992.982027  2.150956e+10
2026-03-11  2035.174154  2.043350e+10


In [25]:
# ADD FEATURE DAILY RETURNS

# pct_change() calculates the percentage change between
# today's price and the previous day's price.
#
# Example:
#
# Yesterday = $100
# Today = $105
#
# Daily Return = 5%
df["daily_return"] = df["Close"].pct_change()

In [ ]:
# ADD FEATURE MOVING AVERAGES

